In [12]:
import requests
import pandas as pd
import time


In [13]:

# ============================================================
# INDIA BOUNDING BOX
# ============================================================
SOUTH = 6.0
WEST = 68.0
NORTH = 37.5
EAST = 97.5

# ============================================================
# OVERPASS SERVERS
# ============================================================
SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]

HEADERS = {
    "User-Agent": "PS162-Industrial-Thermal-Source-Research/1.0"
}


In [14]:

# ============================================================
# TAGS WE WANT
#
# We query these separately so we know exactly why an object
# was included.
# ============================================================
QUERIES = {

    # --------------------------------------------------------
    # GENERAL INDUSTRIAL FACILITIES
    # --------------------------------------------------------
    "industrial_facility": f"""
    [out:json][timeout:300];
    (
        nwr["landuse"="industrial"]({SOUTH},{WEST},{NORTH},{EAST});
        nwr["industrial"]({SOUTH},{WEST},{NORTH},{EAST});
        nwr["man_made"="works"]({SOUTH},{WEST},{NORTH},{EAST});
    );
    out center tags;
    """,

    # --------------------------------------------------------
    # REFINERIES
    # --------------------------------------------------------
    "refinery": f"""
    [out:json][timeout:300];
    (
        nwr["industrial"="refinery"]({SOUTH},{WEST},{NORTH},{EAST});
    );
    out center tags;
    """,

    # --------------------------------------------------------
    # POWER PLANTS
    # --------------------------------------------------------
    "power_plant": f"""
    [out:json][timeout:300];
    (
        nwr["power"="plant"]({SOUTH},{WEST},{NORTH},{EAST});
    );
    out center tags;
    """,

    # --------------------------------------------------------
    # MINES
    # --------------------------------------------------------
    "mine": f"""
    [out:json][timeout:300];
    (
        nwr["industrial"="mine"]({SOUTH},{WEST},{NORTH},{EAST});
        nwr["man_made"="mineshaft"]({SOUTH},{WEST},{NORTH},{EAST});
        nwr["man_made"="adit"]({SOUTH},{WEST},{NORTH},{EAST});
    );
    out center tags;
    """,

    # --------------------------------------------------------
    # QUARRIES / SURFACE MINING
    # --------------------------------------------------------
    "quarry": f"""
    [out:json][timeout:300];
    (
        nwr["landuse"="quarry"]({SOUTH},{WEST},{NORTH},{EAST});
    );
    out center tags;
    """,

    # --------------------------------------------------------
    # GAS / OIL INFRASTRUCTURE
    # --------------------------------------------------------
    "oil_gas": f"""
    [out:json][timeout:300];
    (
        nwr["industrial"="oil"]({SOUTH},{WEST},{NORTH},{EAST});
        nwr["industrial"="gas"]({SOUTH},{WEST},{NORTH},{EAST});
        nwr["utility"="gas"]({SOUTH},{WEST},{NORTH},{EAST});
    );
    out center tags;
    """,

    # --------------------------------------------------------
    # GAS FLARES
    # --------------------------------------------------------
    "gas_flare": f"""
    [out:json][timeout:300];
    (
        nwr["man_made"="flare"]({SOUTH},{WEST},{NORTH},{EAST});
    );
    out center tags;
    """,

    # --------------------------------------------------------
    # VOLCANOES
    # --------------------------------------------------------
    "volcano": f"""
    [out:json][timeout:300];
    (
        nwr["natural"="volcano"]({SOUTH},{WEST},{NORTH},{EAST});
        nwr["geological"="volcanic_vent"]({SOUTH},{WEST},{NORTH},{EAST});
    );
    out center tags;
    """
}



In [15]:

# ============================================================
# OVERPASS REQUEST FUNCTION
# ============================================================
def get_overpass_data(query):

    for server in SERVERS:

        print(f"\nTrying server: {server}")

        try:

            response = requests.post(
                server,
                data={"data": query},
                headers=HEADERS,
                timeout=360
            )

            print("HTTP status:", response.status_code)

            response.raise_for_status()

            return response.json()

        except requests.exceptions.RequestException as e:

            print("Request failed:", e)
            time.sleep(5)

    return None



In [17]:

# ============================================================
# DOWNLOAD + PROCESS
# ============================================================
rows = []


for category, query in QUERIES.items():

    print("\n" + "=" * 70)
    print(f"DOWNLOADING: {category}")
    print("=" * 70)

    data = get_overpass_data(query)

    if data is None:
        print(f"Could not download {category}")
        continue

    elements = data.get("elements", [])

    print("Objects received:", len(elements))

    for element in elements:

        tags = element.get("tags", {})

        # -----------------------------------------------
        # Coordinates
        # -----------------------------------------------

        if element["type"] == "node":

            lat = element.get("lat")
            lon = element.get("lon")

        else:

            center = element.get("center", {})

            lat = center.get("lat")
            lon = center.get("lon")

        # -----------------------------------------------
        # Store useful information
        # -----------------------------------------------

        rows.append({

            "osm_id": element["id"],

            "osm_type": element["type"],

            "latitude": lat,

            "longitude": lon,

            "name": tags.get("name"),

            "osm_category": category,

            "industrial": tags.get("industrial"),

            "landuse": tags.get("landuse"),

            "power": tags.get("power"),

            "plant_source": tags.get("plant:source"),

            "man_made": tags.get("man_made"),

            "natural": tags.get("natural"),

            "geological": tags.get("geological"),

            "volcano_status": tags.get("volcano:status"),

            "volcano_type": tags.get("volcano:type"),

            "operator": tags.get("operator"),

            "resource": tags.get("resource"),

            "product": tags.get("product"),

            "utility": tags.get("utility"),

            "wikidata": tags.get("wikidata"),

            "raw_tags": str(tags)
        })




DOWNLOADING: industrial_facility

Trying server: https://overpass-api.de/api/interpreter
HTTP status: 200
Objects received: 48618

DOWNLOADING: refinery

Trying server: https://overpass-api.de/api/interpreter
HTTP status: 200
Objects received: 39

DOWNLOADING: power_plant

Trying server: https://overpass-api.de/api/interpreter
HTTP status: 429
Request failed: 429 Client Error: Too Many Requests for url: https://overpass-api.de/api/interpreter

Trying server: https://maps.mail.ru/osm/tools/overpass/api/interpreter
Request failed: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

Trying server: https://overpass.kumi.systems/api/interpreter
HTTP status: 200
Objects received: 7108

DOWNLOADING: mine

Trying server: https://overpass-api.de/api/interpreter
HTTP status: 504
Request failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter

Trying server: https://maps.mail.ru/osm/tools/overpass/api/interpreter
Request failed: ('Co

In [18]:

# ============================================================
# DATAFRAME
# ============================================================
df = pd.DataFrame(rows)

if df.empty:

    raise RuntimeError(
        "No data was downloaded from Overpass."
    )


# ============================================================
# REMOVE OBJECTS WITHOUT COORDINATES
# ============================================================
df = df.dropna(
    subset=["latitude", "longitude"]
)



In [19]:

# ============================================================
# REMOVE DUPLICATES
#
# Same OSM feature may have matched more than one query.
# ============================================================
df = df.drop_duplicates(
    subset=["osm_type", "osm_id"]
)


# ============================================================
# SORT
# ============================================================
df = df.sort_values(
    by=["osm_category", "name"],
    na_position="last"
)


# ============================================================
# SAVE
# ============================================================
OUTPUT_FILE = "india_industrial_thermal_sources_osm.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False
)



In [20]:

# ============================================================
# REPORT
# ============================================================
print("\n")
print("=" * 70)
print("DOWNLOAD COMPLETE")
print("=" * 70)

print("Total objects:", len(df))

print("\nObjects by category:")
print(
    df["osm_category"]
    .value_counts()
)

print("\nIndustrial types:")
print(
    df["industrial"]
    .value_counts(dropna=False)
    .head(30)
)

print("\nPower types:")
print(
    df["power"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nPlant sources:")
print(
    df["plant_source"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nVolcano status:")
print(
    df["volcano_status"]
    .value_counts(dropna=False)
)

print("\nSaved to:")
print(OUTPUT_FILE)

print("=" * 70)



DOWNLOAD COMPLETE
Total objects: 67115

Objects by category:
osm_category
industrial_facility    48618
quarry                 12760
power_plant             4828
mine                     741
volcano                   68
oil_gas                   67
gas_flare                 33
Name: count, dtype: int64

Industrial types:
industrial
NaN               58153
brickyard          5138
factory             648
depot               631
brickworks          475
mine                229
cooling             205
warehouse           157
grinding_mill       147
slaughterhouse      129
port                103
scrap_yard           89
rice_mill            76
oil                  72
bus_depot            40
agriculture          39
refinery             39
railway              39
chemical             30
sawmill              30
timber               26
storage              24
textile              21
communication        20
cement               19
manufacturing        19
yes                  19
concrete_plant   